## Install Libraries

Runs the **LLM feedback stage of GRF and nothing else** (architecture §5.1, §6):
load the gold BEL data from Google Drive, ask the feedback model for every
mention's `def`/`syn`/`std` text, and save the answers to **one** JSONL file.

That file is the handoff: `GRF/ranking.ipynb` replays it through the encoder,
the index, fusion and ranking, so the feedback model's weights never share a
GPU with the KB vectors and the retrieval pass never calls an LLM at all.

No encoder and no vector index are loaded here, which is why this notebook
installs neither `sentence-transformers` nor the KB artifact's dependencies:
`transformers` backs `LLM_BACKEND = "local"`, `pyyaml` is needed because the
prompt templates are read from `source/GRF/prompts.yaml`, and `tqdm` because
every long stage reports progress.

Run order: `common/build_KB.ipynb` → `GRF/feedback.ipynb` → `GRF/ranking.ipynb`
→ `GRF/reranking.ipynb`.

Sources: `docs/architecture.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.


In [ ]:
%pip install "tqdm>=4.66" "transformers>=4.44" "pyyaml>=6.0"

## Import Libraries

Third-party imports only. The `common` and `GRF` packages live inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.


In [ ]:
from __future__ import annotations

import gc
import os
import subprocess
import sys
import time
from collections import Counter
from contextlib import contextmanager
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()


@contextmanager
def phase_progress():
    """One tqdm bar per pipeline phase, created when that phase first reports.

    ``run_feedback_batch`` and ``link_batch`` both report
    ``(phase, done, total)``; phases run in order, so each gets its own bar
    instead of one bar counting a mixture of work. A phase whose ``total`` is
    None (the encoder batches internally) shows a running count rather than a
    percentage.
    """
    bars = {}

    def report(phase, done, total=None) -> None:
        bar = bars.get(phase)
        if bar is None:
            bar = tqdm(total=total, desc=phase, unit="item", leave=True)
            bars[phase] = bar
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        for bar in bars.values():
            bar.close()


## Constants and Configuration

All paths, feedback parameters and LLM settings live here.

Drive layout:

- `COMMON_DIR/` — shared root (`DRIVE_ROOT/work/nlp/UMLS/entity_linking`),
- `COMMON_DIR/data/` — shared inputs: the gold BEL JSONL,
- `COMMON_DIR/output/{METHOD}/` — this method's outputs (`METHOD = "grf"`).

`FEEDBACK_TYPES = ("def",)` reproduces the paper's single-component run;
`("syn",)` and `("std",)` select the other components, and
`("def", "syn", "std")` is All-in-One (architecture §6, §7). `N_SYN` is one of
1, 3, 5, 10 and only controls how many synonyms `syn` asks for.

**Feedback model.** `LLM_BACKEND = "local"` loads a HuggingFace causal LM
directly through `transformers` (`LLM_MODEL_LOCAL`: a Hub id or a local
directory) — no API key, no network at inference time once the weights are
cached. `LLM_BACKEND = "api"` uses the OpenAI-compatible HTTP client and needs
`OPENAI_API_KEY`. Either path degrades a failed kind to `Q_0` instead of
stopping the run (architecture §6), and the failure is recorded in the bundle.

The default feedback model is `Qwen/Qwen3-32B`, decoded in batches of
`LLM_LOCAL_BATCH_SIZE` prompts: decoding is memory-bandwidth bound, so one
padded batch costs far less than one call per prompt. Use 8 on an 80 GB GPU, 4
when the 32B weights share the card with anything else.

**Two output files, deliberately.** `FEEDBACK_BUNDLE_PATH` is the handoff read
by `ranking.ipynb` — one record per mention, pinned to
`<model>|<prompt digest>`. `FEEDBACK_CACHE_PATH` is the append-only per-kind
cache (`JsonlFeedbackCache`, same namespace): a re-run — or a Colab that died
mid-run — reloads every text it already paid for instead of generating it
twice. Delete the cache file to force regeneration.


In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive layout
#   COMMON_DIR/                     shared root
#   COMMON_DIR/data/                shared inputs (gold data)
#   COMMON_DIR/output/{METHOD}/     this method's outputs
# ---------------------------------------------------------------------------
METHOD = "grf"
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
COMMON_DIR = DRIVE_ROOT / "work" / "nlp" / "UMLS" / "entity_linking"
DRIVE_DATA_DIR = COMMON_DIR / "data"
OUTPUT_DIR = COMMON_DIR / "output" / METHOD

GOLD_DATA_PATH = DRIVE_DATA_DIR / "test.jsonl"                  # gold BEL JSONL
FEEDBACK_BUNDLE_PATH = OUTPUT_DIR / "grf_feedback_bundle.jsonl"  # handoff to ranking.ipynb
FEEDBACK_CACHE_PATH = OUTPUT_DIR / "grf_feedback.jsonl"          # reusable per-kind cache

# ---------------------------------------------------------------------------
# Encoder name: provenance only. This notebook loads no encoder; the config
# requires the field and ranking.ipynb is where the checkpoint is checked
# against the KB artifact.
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

# ---------------------------------------------------------------------------
# GRF feedback parameters (architecture §6, §9)
# ---------------------------------------------------------------------------
FEEDBACK_TYPES = ("def",)       # (), ("def",), ("syn",), ("std",), or all three
N_SYN = 5                       # one of 1, 3, 5, 10
TARGET_KB = "KB"                # KB name used by the `std` instruction

# ---------------------------------------------------------------------------
# Feedback model: local HuggingFace weights (default) or the OpenAI-compatible API
# ---------------------------------------------------------------------------
LLM_BACKEND = "local"           # "local" (transformers) or "api"

# --- "local": loaded through transformers; HF id or a local directory path ---
LLM_MODEL_LOCAL = "Qwen/Qwen3-32B"
LLM_LOCAL_DEVICE = "auto"       # "auto", "cuda", "cpu" or "cuda:0"
LLM_LOCAL_DTYPE = "auto"        # "auto", "float16", "bfloat16" or "float32"
LLM_LOCAL_MAX_NEW_TOKENS = 256  # decode budget per feedback call
LLM_LOCAL_BATCH_SIZE = 4        # prompts decoded together; 8 on an 80 GB GPU
LLM_LOCAL_TEMPERATURE = 0.0     # 0.0 = greedy (deterministic)
LLM_LOCAL_TRUST_REMOTE_CODE = False
LLM_LOCAL_USE_CHAT_TEMPLATE = True   # False = feed the raw prompt

# --- "api": OpenAI-compatible HTTP endpoint; needs OPENAI_API_KEY ---
LLM_MODEL_API = "gpt-4o"
LLM_API_BASE_URL = ""           # empty -> OPENAI_BASE_URL env var / api.openai.com
LLM_API_TIMEOUT_S = 30.0
# Set the key in this runtime (never commit it):
#   import getpass, os; os.environ["OPENAI_API_KEY"] = getpass.getpass()

LLM_ACTIVE_MODEL = LLM_MODEL_LOCAL if LLM_BACKEND == "local" else LLM_MODEL_API

# ---------------------------------------------------------------------------
# Evaluation / runtime
# ---------------------------------------------------------------------------
MAX_SAMPLES = None              # None = all messages; an int truncates for smoke runs
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value

log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"Gold data    : {GOLD_DATA_PATH}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Bundle       : {FEEDBACK_BUNDLE_PATH}")
print(f"Feedback file: {FEEDBACK_CACHE_PATH}")
print(f"GRF settings : feedback={FEEDBACK_TYPES} n_syn={N_SYN} target_kb={TARGET_KB}")
print(f"Feedback LLM : backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL}")
if LLM_BACKEND == "local":
    print(f"LLM batching : {LLM_LOCAL_BATCH_SIZE} prompt(s) per decode")


## Stage 1 — Clone or Update the GRF Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/` and the feedback module
from `source/GRF/`. Re-running this cell updates the existing clone instead of
downloading it again.


In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.data import load_bel_jsonl

from GRF.config import PipelineConfig
from GRF.evaluator import make_requests
from GRF.feedback import (
    FEEDBACK_BUNDLE_FORMAT,
    FeedbackGenerator,
    JsonlFeedbackCache,
    feedback_namespace,
    load_feedback_outcomes,
    run_feedback_batch,
    save_feedback_outcomes,
)
from GRF.hf_llm import TransformersChatClient
from GRF.llm import OpenAICompatibleClient

log(f"Repository ready at {REPO_CLONE_DIR}; common and GRF imported from {SOURCE_DIR}.")


def build_llm_client(backend, model):
    """Local transformers model or the OpenAI-compatible API client."""
    if backend == "local":
        return TransformersChatClient(
            model,
            device=LLM_LOCAL_DEVICE,
            dtype=LLM_LOCAL_DTYPE,
            max_new_tokens=LLM_LOCAL_MAX_NEW_TOKENS,
            temperature=LLM_LOCAL_TEMPERATURE,
            trust_remote_code=LLM_LOCAL_TRUST_REMOTE_CODE,
            use_chat_template=LLM_LOCAL_USE_CHAT_TEMPLATE,
            batch_size=LLM_LOCAL_BATCH_SIZE,
        )
    if backend == "api":
        if not os.environ.get("OPENAI_API_KEY"):
            log(
                "WARNING: OPENAI_API_KEY is not set. API calls will fail and "
                "every feedback kind will degrade to Q_0."
            )
        return OpenAICompatibleClient(
            model, base_url=LLM_API_BASE_URL or None, timeout_s=LLM_API_TIMEOUT_S
        )
    raise ValueError(f"LLM_BACKEND must be 'local' or 'api', got {backend!r}")


log("LLM factory ready; feedback backend=%s." % LLM_BACKEND)


## Stage 2 — Mount Google Drive and Locate the Inputs

The gold data lives on Drive, not in the repository. This stage mounts Drive
and checks the gold path and the two output paths before any model is loaded.


In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gold_path = Path(require_configured(GOLD_DATA_PATH, "GOLD_DATA_PATH"))
bundle_path = Path(require_configured(FEEDBACK_BUNDLE_PATH, "FEEDBACK_BUNDLE_PATH"))
cache_path = Path(require_configured(FEEDBACK_CACHE_PATH, "FEEDBACK_CACHE_PATH"))

if not gold_path.is_file():
    raise FileNotFoundError(f"Gold data not found: {gold_path}")
log(f"Gold data: {gold_path} ({gold_path.stat().st_size / 1024 ** 2:.2f} MB)")

for label, path in (("Bundle output", bundle_path), ("Feedback cache", cache_path)):
    if path.suffix.lower() != ".jsonl":
        raise ValueError(f"{label} must end in .jsonl, got {path.name!r}.")
    log(f"{label}: {path}")


## Stage 3 — Load and Validate the Gold Data

`load_bel_jsonl` validates structure and token spans. Structural problems are
reported as `DataIssue` entries; span issues never drop a sample. Every sample
is used, and the mention/context pair is exactly what the feedback prompts are
built from. The loader reports each record through `progress_callback`, which
drives the bar.


In [ ]:
log(f"Loading gold BEL data from {gold_path} ...")
started = time.time()
with progress_bar(None, "Loading gold records", "record") as report:
    loaded = load_bel_jsonl(gold_path, strict=False, progress_callback=report)
samples = list(loaded.samples)
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]

log(
    f"Loaded {len(loaded.samples):,} mention samples "
    f"({len(samples):,} selected) in {time.time() - started:.1f}s."
)
if loaded.issues:
    log(f"WARNING: {len(loaded.issues):,} data issue(s); first: {loaded.issues[0]}")
if not samples:
    raise ValueError(f"{gold_path} produced no usable mention samples.")


## Stage 4 — Configure the Feedback Generator

`make_requests(samples, config)` builds one request per mention from
`PipelineConfig`; only `feedback_types`, `n_syn` and `target_kb` reach this
stage. `fusion`, `rerank`, `top_k_*` and `index_kb` belong to `ranking.ipynb`.

The generator keeps a `JsonlFeedbackCache` at `FEEDBACK_CACHE_PATH`,
namespaced by `<model>|<prompt digest>`: entries on disk are reused when the
model and `prompts.yaml` are unchanged, and ignored when either changed, so a
stale answer can never be attributed to a new prompt. A failed kind is
collected as a `FeedbackFailure` — it does not stop the run.


In [ ]:
config = PipelineConfig(
    encoder=ENCODER_MODEL,      # provenance only; no encoder is loaded here
    feedback_types=tuple(FEEDBACK_TYPES),
    n_syn=N_SYN,
    target_kb=TARGET_KB,
    llm_model=LLM_ACTIVE_MODEL,
)

requests = make_requests(samples, config)
log(f"{len(requests):,} feedback request(s) built: kinds={config.feedback_types or 'none (baseline)'}.")

llm = None
feedback_generator = None
if config.feedback_types:
    llm = build_llm_client(LLM_BACKEND, LLM_ACTIVE_MODEL)
    feedback_cache = JsonlFeedbackCache(
        cache_path, namespace=feedback_namespace(LLM_ACTIVE_MODEL)
    )
    log(
        f"Feedback cache: {cache_path} "
        f"({len(feedback_cache):,} entries reusable for this model + prompt)."
    )
    feedback_generator = FeedbackGenerator(llm, cache=feedback_cache)
    log(
        f"Feedback LLM: backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL!r} "
        "(weights load lazily on the first call)."
    )
else:
    log(
        "WARNING: FEEDBACK_TYPES is empty, so this notebook generates nothing and "
        "ranking.ipynb will reproduce the no-GRF baseline through the bundle."
    )


## Stage 5 — Generate the Feedback, Then Free the Model

`run_feedback_batch` is the LLM half of the pipeline alone: every prompt of the
batch goes to the client at once, so a local model decodes one padded batch
instead of one prompt at a time, and each mention's per-kind outcome comes back
in request order. Only this stage needs the language model — no encoder, no
vector index — so the weights are released and the CUDA cache emptied here,
before anything else claims the GPU.

The bar advances when the batch returns: a local model decodes the whole batch
inside one `generate` call, so there is no finer-grained progress to report
than "one call is running".

`JsonlFeedbackCache` appends each text to the cache file as it is produced, so
an interrupted run keeps what it already paid for and a re-run reloads instead
of regenerating. That is why re-running this cell after a crash is cheap.


In [ ]:
started = time.time()
if feedback_generator is None:
    feedback_outcomes = [((), ()) for _ in requests]
else:
    with phase_progress() as report:
        feedback_outcomes = run_feedback_batch(
            feedback_generator, requests, config.target_kb, progress_callback=report
        )

kind_counts = Counter(
    feedback.kind for feedbacks, _ in feedback_outcomes for feedback in feedbacks
)
failure_counts = Counter(
    failure.kind for _, failures in feedback_outcomes for failure in failures
)
text_counts = sum(
    len(feedback.texts) for feedbacks, _ in feedback_outcomes for feedback in feedbacks
)
log(
    f"Feedback ready for {len(requests):,} mentions in "
    f"{time.time() - started:.1f}s: {text_counts:,} text(s) over "
    f"{dict(kind_counts) or '{}'}; {sum(failure_counts.values()):,} kind(s) "
    f"degraded to Q_0 {dict(failure_counts) if failure_counts else ''}."
)
if failure_counts:
    first = next(
        failure
        for _, failures in feedback_outcomes
        for failure in failures
    )
    log(f"First failure: {first.kind}: {first.reason}")

if llm is not None and hasattr(llm, "unload"):
    llm.unload()
    log("Feedback model weights unloaded.")
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass


## Stage 6 — Save the Feedback Bundle

`save_feedback_outcomes` writes one JSONL record per mention: the mention, its
context, the gold token span, and the per-kind texts with the failures. It is
written in request order and pinned to `feedback_namespace(LLM_ACTIVE_MODEL)`,
i.e. `<model>|<prompt digest>` — `ranking.ipynb` passes the same namespace, so
switching model or editing `prompts.yaml` is a loud error there instead of a
silent replay of answers the new prompt would not have given.

The file is reloaded immediately to prove it is usable and to show what
`ranking.ipynb` will see. The reloaded requests ask for no feedback at all
(`feedback_types=()`): retrieval must not need a generator.


In [ ]:
namespace = feedback_namespace(LLM_ACTIVE_MODEL)
log(f"Saving {len(requests):,} feedback records to {bundle_path} (ns={namespace}) ...")
started = time.time()
with progress_bar(len(requests), "Saving feedback records", "record") as report:
    save_feedback_outcomes(
        bundle_path,
        requests,
        feedback_outcomes,
        namespace=namespace,
        progress_callback=report,
    )
log(
    f"Saved {len(requests):,} records "
    f"({bundle_path.stat().st_size / 1024 ** 2:.2f} MB) in {time.time() - started:.1f}s."
)

log("Reloading the bundle to verify it ...")
with progress_bar(None, "Reloading feedback records", "record") as report:
    reloaded_requests, reloaded_outcomes = load_feedback_outcomes(
        bundle_path, namespace=namespace, progress_callback=report
    )
assert len(reloaded_requests) == len(requests), (len(reloaded_requests), len(requests))
assert [request.mention for request in reloaded_requests] == [
    request.mention for request in requests
]
assert [request.start_token for request in reloaded_requests] == [
    request.start_token for request in requests
]
assert [
    [(feedback.kind, feedback.texts) for feedback in feedbacks]
    for feedbacks, _ in reloaded_outcomes
] == [
    [(feedback.kind, feedback.texts) for feedback in feedbacks]
    for feedbacks, _ in feedback_outcomes
]
assert all(
    request.feedback_types == () for request in reloaded_requests
), "a reloaded request must ask for no feedback"
log(
    f"Verified: {len(reloaded_requests):,} requests and their feedback reloaded "
    f"({FEEDBACK_BUNDLE_FORMAT}); kinds per mention are read from this file."
)


## Stage 7 — Release Memory

The bundle and the cache are on Drive and the model is already unloaded, so
nothing else is held. Drop the remaining objects and empty the CUDA cache.
Re-run from stage 3 if you need the live generator again.

Next: `source/notebooks/GRF/ranking.ipynb`.


In [ ]:
if "llm" in globals() and llm is not None and hasattr(llm, "unload"):
    llm.unload()
    log("Local feedback model weights unloaded.")

for name in (
    "llm",
    "feedback_generator",
    "feedback_cache",
    "feedback_outcomes",
    "loaded",
    "samples",
    "requests",
    "reloaded_requests",
    "reloaded_outcomes",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("Feedback memory released. Next: source/notebooks/GRF/ranking.ipynb")
